# 03 — Train Autoencoder
Unsupervised anomaly detection: train on normal traffic only.

In [1]:
import sys
sys.path.insert(0, '..')

import pickle
import torch

from src.config import (
    DEVICE, AE_LATENT_DIM, AE_HIDDEN_DIMS, AE_LR, AE_EPOCHS, AE_PATIENCE,
    AUTOENCODER_CHECKPOINT, PROCESSED_DIR, BATCH_SIZE
)
from src.models import Autoencoder
from src.dataset import make_loader
from src.train_utils import train_autoencoder
from src.visualize import plot_loss_curves

print(f'Device: {DEVICE}')

Device: cuda


## Load preprocessed data

In [2]:
with open(PROCESSED_DIR / 'data.pkl', 'rb') as f:
    data = pickle.load(f)

# Train autoencoder only on NORMAL traffic (unsupervised)
X_train_normal = data['X_train'][data['normal_mask_train']]
X_val = data['X_val']  # validate on all (to see reconstruction error on attacks too)
print(f'Normal training samples: {len(X_train_normal):,}')
print(f'Validation samples     : {len(X_val):,}')
input_dim = X_train_normal.shape[1]
print(f'Input dimension        : {input_dim}')

Normal training samples: 60,608
Validation samples     : 12,598
Input dimension        : 41


## Build model & loaders

In [3]:
model = Autoencoder(input_dim=input_dim, hidden_dims=AE_HIDDEN_DIMS, latent_dim=AE_LATENT_DIM)
model = model.to(DEVICE)
print(model)

optimizer = torch.optim.Adam(model.parameters(), lr=AE_LR)

train_loader = make_loader(X_train_normal, shuffle=True, batch_size=BATCH_SIZE)
val_loader   = make_loader(X_val, data['y_bin_val'], shuffle=False, batch_size=BATCH_SIZE)

Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=41, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=48, bias=True)
    (4): BatchNorm1d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Linear(in_features=48, out_features=32, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=32, out_features=48, bias=True)
    (1): BatchNorm1d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=48, out_features=64, bias=True)
    (4): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=41, bias=True)
  )
)


## Train

In [4]:
history = train_autoencoder(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=DEVICE,
    epochs=AE_EPOCHS,
    patience=AE_PATIENCE,
    checkpoint_path=AUTOENCODER_CHECKPOINT,
)
print(f'\nBest checkpoint: {AUTOENCODER_CHECKPOINT}')

Epoch   1/50  train_loss=0.444087  val_loss=0.436184


Epoch   2/50  train_loss=0.281033  val_loss=0.339253


Epoch   3/50  train_loss=0.235632  val_loss=0.377039


Epoch   4/50  train_loss=0.213990  val_loss=0.357001


Epoch   5/50  train_loss=0.198917  val_loss=0.302828


...

Epoch  14/50  train_loss=0.122471  val_loss=0.287096


Epoch  15/50  train_loss=0.109076  val_loss=0.271455


Epoch  16/50  train_loss=0.100740  val_loss=0.339659


Epoch  17/50  train_loss=0.097078  val_loss=0.267717


Epoch  18/50  train_loss=0.095594  val_loss=0.348853
Early stopping at epoch 18

Best checkpoint: /home/abzy/dev/aitu/masters/trimester3/aitu-multi-agent-systems/notebooks/../checkpoints/autoencoder_best.pt


## Loss curves

In [5]:
plot_loss_curves(history, title='Autoencoder Loss Curves', filename='ae_loss_curves.png')

Saved: /home/abzy/dev/aitu/masters/trimester3/aitu-multi-agent-systems/notebooks/../artifacts/ae_loss_curves.png


PosixPath('/home/abzy/dev/aitu/masters/trimester3/aitu-multi-agent-systems/notebooks/../artifacts/ae_loss_curves.png')